# Connective-field figures

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import cortex

from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO_ROOT))
from shared.paths import DATA, OUTPUTS, ROOT
VICSOMPY_REPO = os.environ.get("VICSOMPY_REPO", str(ROOT / "Vicarious_somatotopy"))
if VICSOMPY_REPO not in sys.path:
    sys.path.insert(0, VICSOMPY_REPO)

from cf_modeling.cf_naming import cf_model_map_stems, per_subject_mean_stem, qualify_lboe
from vicsompy.vis import Plot, basic_plot
from vicsompy.utils import load_plot_pkg_yaml

PYCORTEX_STORE = str(DATA / "vicsompy_data")
new_db = cortex.database.Database(PYCORTEX_STORE)
cortex.db = new_db
cortex.database.default_filestore = new_db
for _mod_name, _mod in sys.modules.items():
    if _mod_name.startswith("cortex") and hasattr(_mod, "db"):
        _mod.db = new_db
cortex.options.config.set("basic", "filestore", PYCORTEX_STORE)
cortex.options.config.set("webgl", "colormaps", os.path.join(sys.prefix, "share", "pycortex", "colormaps"))
CX_SUB = list(cortex.db.subjects.keys())[0]

In [ ]:
MODE     = "group_average"
OUT_BASE = str(OUTPUTS / "cf_modeling")
LBOE     = 200

TEMPLATE_CIFTI = str(
    DATA / "preprocessed/average_sub/hedger_sg_psc/group_average_hedger_sg_psc_cortex_59k.dtseries.nii"
)

bm_axis   = nib.load(TEMPLATE_CIFTI).header.get_axis(1)
plotviews = load_plot_pkg_yaml()['plotviews']

In [ ]:
def to_sphere(arr):
    sphere = np.zeros(118584, dtype=np.float32)
    pos = 0
    for name, _, model in bm_axis.iter_structures():
        n = len(model.vertex)
        if "CORTEX_LEFT" in name:
            sphere[model.vertex] = arr[pos:pos+n]
        elif "CORTEX_RIGHT" in name:
            sphere[59292 + model.vertex] = arr[pos:pos+n]
        else:
            pos += n; continue
        pos += n
    return sphere


def load_pair(roi_a, roi_b):
    roi_a, roi_b = qualify_lboe(roi_a, LBOE), qualify_lboe(roi_b, LBOE)
    stems = cf_model_map_stems(roi_a, roi_b)
    roi_root = f"{OUT_BASE}/{MODE}/{roi_a}_{roi_b}"
    prep_dir = f"{roi_root}/prep" if MODE == "group_average" else f"{roi_root}/group"

    def _load(key):
        stem = stems[key] if MODE == "group_average" else per_subject_mean_stem(stems[key])
        arr = np.load(os.path.join(prep_dir, f"{stem}.npy"))
        return arr

    d = dict(roi_root=roi_root, prep_dir=prep_dir, roi_a=roi_a, roi_b=roi_b)
    d['R2_a']      = _load("split_r2_a")
    d['R2_b']      = _load("split_r2_b")
    d['R2_a_nc']   = _load("null_corrected_split_r2_a")
    d['R2_b_nc']   = _load("null_corrected_split_r2_b")
    d['R2_full']   = _load("full_r2")
    d['product']   = _load("joint_split_r2_geomean")
    d['product_nc'] = _load("joint_null_corrected_split_r2_geomean")
    return d


_CMAP_PNG = os.path.join(sys.prefix, "share/pycortex/colormaps/PU_RdBu_covar_alpha.png")
_SURF_DIR = f"{PYCORTEX_STORE}/{CX_SUB}/surfaces"


def plot_fig3a(roi_a, roi_b, sph_a_pos, sph_b_pos, fig_dir):
    cfg = plotviews['variance_decomp_full'].copy()
    cfg['subject'] = CX_SUB
    cfg.pop('ROIS', None)
    fig = Plot(**cfg).uber_plot(dat=sph_b_pos, dat2=sph_a_pos)
    out = f"{fig_dir}/fig3a_{roi_a}_{roi_b}_2d_colormap.png"
    fig.savefig(out, bbox_inches='tight', dpi=cfg.get('dpi', 300))
    plt.show()


def plot_inflated(roi_a, roi_b, sph_a_pos, sph_b_pos, fig_dir):
    import matplotlib.image as mpimg
    from matplotlib.collections import PolyCollection

    tex = mpimg.imread(_CMAP_PNG)
    H, W = tex.shape[:2]
    vmax = 0.4
    xi = np.clip((np.nan_to_num(sph_b_pos) / vmax * (W - 1)).astype(int), 0, W - 1)
    yi = np.clip((np.nan_to_num(sph_a_pos) / vmax * (H - 1)).astype(int), 0, H - 1)
    yi = (H - 1) - yi
    rgba_all = tex[yi, xi]

    panels = [
        ("L lateral", "inflated_lh", 0,     +1, False),
        ("L medial",  "inflated_lh", 0,     -1, True),
        ("R lateral", "inflated_rh", 59292, -1, True),
        ("R medial",  "inflated_rh", 59292, +1, False),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(14, 4), dpi=150)
    for ax, (label, fname, offset, depth_sign, flip_u) in zip(axes, panels):
        surf   = nib.load(f"{_SURF_DIR}/{fname}.gii")
        coords = surf.darrays[0].data.astype(np.float32)
        faces  = surf.darrays[1].data.astype(np.int32)
        rgba_h = rgba_all[offset:offset + 59292]

        centroid_x = coords[faces].mean(axis=1)[:, 0]
        order = np.argsort(-centroid_x * depth_sign)

        u = -coords[:, 1] if flip_u else coords[:, 1]
        v = coords[:, 2]

        verts   = np.stack([u[faces[order]], v[faces[order]]], axis=-1)
        fcolors = rgba_h[faces[order]].mean(axis=1)

        coll = PolyCollection(
            verts, facecolors=fcolors,
            edgecolors='none', linewidths=0, rasterized=True,
        )
        ax.add_collection(coll)
        ax.set_xlim(u.min(), u.max())
        ax.set_ylim(v.min(), v.max())
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(label, fontsize=9, pad=2)

    fig.suptitle(f"{roi_a} (red) × {roi_b} (blue)  [inflated]", fontsize=10)
    plt.tight_layout()
    out = f"{fig_dir}/inflated_{roi_a}_{roi_b}_2d_colormap.png"
    fig.savefig(out, bbox_inches='tight', dpi=150)
    plt.show()


def print_stats_table(roi_a, roi_b, d):
    print(f"\n{'Map':<22}  {'Mean':>8}  {'Frac>0':>7}  {'Max':>8}  {'Min':>8}")
    print("-" * 60)
    for name, arr in [
        (f"R2_{roi_a}",    d['R2_a']),
        (f"R2_{roi_b}",    d['R2_b']),
        (f"R2_{roi_a}_nc", d['R2_a_nc']),
        (f"R2_{roi_b}_nc", d['R2_b_nc']),
        ("R2_full",        d['R2_full']),
        ("product_map",    d['product']),
        ("product_map_nc", d['product_nc']),
    ]:
        print(f"{name:<22}  {arr.mean():>8.4f}  {np.mean(arr>0):>7.1%}"
              f"  {arr.max():>8.4f}  {arr.min():>8.4f}")


def plot_all_for_pair(roi_a, roi_b):
    d = load_pair(roi_a, roi_b)
    roi_a, roi_b = d['roi_a'], d['roi_b']
    fig_dir = f"{d['roi_root']}/figures"
    os.makedirs(fig_dir, exist_ok=True)

    sph_a_pos = np.clip(to_sphere(d['R2_a']), 0, None)
    sph_b_pos = np.clip(to_sphere(d['R2_b']), 0, None)

    print_stats_table(roi_a, roi_b, d)
    plot_fig3a(roi_a, roi_b, sph_a_pos, sph_b_pos, fig_dir)
    plot_inflated(roi_a, roi_b, sph_a_pos, sph_b_pos, fig_dir)

## 3b × V1

In [ ]:
plot_all_for_pair("3b", "V1")

## A1 × V1

In [ ]:
plot_all_for_pair("A1", "V1")

## A5 × FFC

In [ ]:
plot_all_for_pair("A5", "FFC")